<a href="https://colab.research.google.com/github/droyktton/ICNPG/blob/master/Clases/Colaboratory/cudaC_en_colabs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%html
<marquee style='width: 100%; color: blue;'><b>ICNPG en Google Colaboratory</b></marquee>


Hola!, Bueno, aqui va un ejemplito de como correr codigo CUDA C/C++ en colabs. No se olviden de Runtime-> Change Runtime Type -> GPU. Para que funque a cada linea la tienen que ejecutar con un Shift-Enter o Ctrl-Enter o el botoncito de play.

miremos que version de nvcc tenemos

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


A ver que placa nos toco...

In [3]:
!nvidia-smi

Mon Apr  3 13:09:36 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   70C    P8    11W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

lindas GPUs!!. Ahora, para poder correr Cuda C/C++ lo que yo hice fue, primero:

In [ ]:
!pip install git+git://github.com/andreinechaev/nvcc4jupyter.git


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Cloning git://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-kv4b9_7z
  Running command git clone --filter=blob:none --quiet git://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-kv4b9_7z


y luego:

In [ ]:
%load_ext nvcc_plugin


created output directory at /content/src
Out bin /content/result.out


Probemos ahora un programita simple. Notar que al principio tiene que tener un %%cu

In [1]:
%%writefile holamundo.cu
#include <stdio.h>
#include <stdlib.h>
__global__ void add(int *a, int *b, int *c) {
printf("hola mundo\n");
*c = *a + *b;
}
int main() {
int a, b, c;
// host copies of variables a, b & c
int *d_a, *d_b, *d_c;
// device copies of variables a, b & c
int size = sizeof(int);
// Allocate space for device copies of a, b, c
cudaMalloc((void **)&d_a, size);
cudaMalloc((void **)&d_b, size);
cudaMalloc((void **)&d_c, size);
// Setup input values  
c = 0;
a = 3;
b = 5;
// Copy inputs to device
cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);
// Launch add() kernel on GPU
add<<<1,1>>>(d_a, d_b, d_c);
// Copy result back to host
cudaError err = cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
  if(err!=cudaSuccess) {
      printf("CUDA error copying to Host: %s\n", cudaGetErrorString(err));
  }
printf("result is %d\n",c);
// Cleanup
cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);
return 0;
}

Writing holamundo.cu


In [3]:
!nvcc /content/holamundo.cu -o holamundo

In [6]:
!/content/holamundo

hola mundo
result is 8


Que les parece? Funca? Bueno, a experimentar mas!. Ahi va uno con thrust. Vamos a hacer la reducción $\sum_{i=0}^{9} 1$

In [8]:
%%writefile suma.cu
#include<thrust/device_vector.h>
#include<iostream>
int main(){
  thrust::device_vector<float> vec(10,1.0);
  std::cout << "la suma de los elementos es = " << thrust::reduce(vec.begin(),vec.end()) << std::endl;
  return 0;    
}

Writing suma.cu


In [9]:
!nvcc /content/suma.cu -o suma; /content/suma

la suma de los elementos es = 10


Funciona thrust. Ahora, a donde va a parar un archivo de output?

In [10]:
%%writefile sumafile.cu
#include<thrust/device_vector.h>
#include<fstream>
int main(){
  std::ofstream fout("output.dat");
  thrust::device_vector<float> vec(10,1.0);
  fout << "la suma de los elementos es =" << thrust::reduce(vec.begin(),vec.end()) << std::endl;
  return 0;    
}

Writing sumafile.cu


In [12]:
!nvcc /content/sumafile.cu -o sumafile; /content/sumafile

Si se fijan a la izquierda, hay un botoncito image.png. Si lo aprietan veran el archivo de output.

Tambien se pueden correr comandos de bash como este:

In [13]:
!for((i=0;i<10;i++)); do printf $i; done

0123456789

Lo mas directo es correr python

In [15]:
# Python 'Hello world' program

print("Hola Mundo")

Hola Mundo
